# GF180MCU-D Analog Agent Setup

This notebook is the setup companion for `share_GF180_Analog`. Open it from the host filesystem, then connect it to the Jupyter kernel served by the IIC-OSIC-TOOLS Docker container. The cells execute inside the container, so paths must be visible to that kernel.


In [ ]:
from pathlib import Path
import json, os, shutil, signal, subprocess, sys, time, urllib.request

def find_share_root(start: Path) -> Path:
    start = start.resolve()
    candidates = []
    for parent in [start, *start.parents]:
        candidates.extend([
            parent,
            parent / 'share_GF180_Analog',
            parent / 'LLM_Amplifier_Sizing' / 'share_GF180_Analog',
        ])
    for candidate in candidates:
        if candidate.name == 'share_GF180_Analog' and (candidate / 'CircuitCollector').is_dir() and (candidate / 'AnalogAgent').is_dir():
            return candidate
    raise FileNotFoundError(
        'Could not find share_GF180_Analog. Set SHARE_ROOT manually to the container-visible '
        'path that contains CircuitCollector/ and AnalogAgent/. Current kernel cwd: '
        f'{start}'
    )

CWD = Path.cwd().resolve()
SHARE_ROOT = find_share_root(CWD)
CC_REPO = SHARE_ROOT / 'CircuitCollector'
CC_ROOT = CC_REPO / 'CircuitCollector'
RFPA_SKILL_ROOT = SHARE_ROOT / 'AnalogAgent' / 'skills' / 'rf-power-amplifier'
ANALOG_AMP_SKILL_ROOT = SHARE_ROOT / 'AnalogAgent' / 'skills' / 'analog-amplifier'
SKILL_ROOT = RFPA_SKILL_ROOT
PDK_SRC = None  # Optional: set Path('/container/path/to/gf180mcuD') to override auto-detection.
PREFERRED_API_PORT = 8001
API_URL = f'http://127.0.0.1:{PREFERRED_API_PORT}'

print('kernel cwd:', CWD)
print('share root:', SHARE_ROOT)
print('CircuitCollector repo:', CC_REPO)
print('CircuitCollector package root:', CC_ROOT)
print('RFPA skill root:', RFPA_SKILL_ROOT)
print('Analog amplifier skill root:', ANALOG_AMP_SKILL_ROOT)


## Link the PDK

The GF180 PDK is intentionally not included in the shared folder. Link the installed in-container PDK into the path expected by the TOML configs.


In [ ]:
pdk_dest = CC_ROOT / 'PDK' / 'gf180mcuD'
required_pdk_files = [
    Path('libs.tech/ngspice/design.ngspice'),
    Path('libs.tech/ngspice/sm141064.ngspice'),
]

def safe_is_file(path: Path) -> bool:
    try:
        return path.is_file()
    except OSError:
        return False

def safe_exists(path: Path) -> bool:
    try:
        return path.exists()
    except OSError:
        return False

def safe_is_dir(path: Path) -> bool:
    try:
        return path.is_dir()
    except OSError:
        return False

def safe_resolve(path: Path) -> Path:
    try:
        return path.resolve()
    except OSError:
        return path

def pdk_is_valid(path: Path | None) -> bool:
    if path is None:
        return False
    return all(safe_is_file(path / rel) for rel in required_pdk_files)

def is_placeholder_pdk_dir(path: Path) -> bool:
    if not safe_is_dir(path) or path.is_symlink():
        return False
    try:
        entries = {p.name for p in path.iterdir()}
    except OSError:
        return False
    return entries <= {'README.md'}

def remove_path_entry(path: Path) -> None:
    try:
        path.unlink()
    except OSError:
        subprocess.run(['rm', '-f', str(path)], check=True)

def remove_invalid_pdk_dest(path: Path) -> None:
    if path.is_symlink():
        try:
            target = os.readlink(path)
            print('Removing stale PDK symlink:', path, '->', target)
        except OSError:
            print('Removing stale PDK symlink:', path)
        remove_path_entry(path)
    elif is_placeholder_pdk_dir(path):
        print('Removing placeholder PDK directory:', path)
        shutil.rmtree(path)

def find_gf180mcuD_pdk() -> Path | None:
    candidates = []
    env_candidates = [os.environ.get('GF180_PDK'), os.environ.get('PDK_ROOT')]
    candidates.extend(Path(p) for p in env_candidates if p)
    candidates.extend([
        Path('/foss/pdks/gf180mcuD'),
        Path('/foss/pdks/gf180mcu/gf180mcuD'),
        Path('/foss/pdks/ciel/gf180mcu/gf180mcuD'),
    ])
    for pattern in [
        '/foss/pdks/ciel/gf180mcu/versions/*/gf180mcuD',
        '/foss/pdks/gf180mcu/versions/*/gf180mcuD',
        '/headless/pdks/ciel/gf180mcu/versions/*/gf180mcuD',
    ]:
        candidates.extend(sorted(Path('/').glob(pattern.lstrip('/'))))
    seen = set()
    for candidate in candidates:
        candidate = safe_resolve(candidate) if safe_exists(candidate) else candidate
        if candidate in seen:
            continue
        seen.add(candidate)
        if pdk_is_valid(candidate):
            return candidate
    return None

pdk_dest.parent.mkdir(parents=True, exist_ok=True)
if pdk_is_valid(pdk_dest):
    print('GF180 PDK already valid:', pdk_dest)
else:
    if safe_exists(pdk_dest) or pdk_dest.is_symlink():
        remove_invalid_pdk_dest(pdk_dest)

    if safe_exists(pdk_dest) or pdk_dest.is_symlink():
        missing = [str(rel) for rel in required_pdk_files if not safe_is_file(pdk_dest / rel)]
        raise FileExistsError(
            f'PDK destination exists but is incomplete: {pdk_dest}. Missing: {missing}. '
            'Move it aside or set PDK_SRC to the real GF180MCU-D PDK path.'
        )

    selected_pdk = PDK_SRC if PDK_SRC is not None else find_gf180mcuD_pdk()
    if selected_pdk is None or not pdk_is_valid(Path(selected_pdk)):
        checked = [
            '/foss/pdks/ciel/gf180mcu/versions/*/gf180mcuD',
            '/foss/pdks/gf180mcu/versions/*/gf180mcuD',
            '/foss/pdks/gf180mcuD',
        ]
        raise FileNotFoundError(
            'Could not auto-detect a valid GF180MCU-D PDK. Set PDK_SRC in the first cell '
            'to the container-visible gf180mcuD directory. Checked common patterns: '
            f'{checked}'
        )
    selected_pdk = Path(selected_pdk)
    pdk_dest.symlink_to(selected_pdk, target_is_directory=True)
    print('Linked', pdk_dest, '->', selected_pdk)

print('PDK design file:', pdk_dest / 'libs.tech/ngspice/design.ngspice')
print('PDK model file :', pdk_dest / 'libs.tech/ngspice/sm141064.ngspice')


## Install and Start the API

Run this inside the Docker container or another environment with Python 3.11+, ngspice, and the GF180 PDK available.


In [ ]:
setup_py = CC_REPO / 'setup.py'
if not setup_py.is_file():
    raise FileNotFoundError(f'CircuitCollector setup.py not found at {setup_py}. Check SHARE_ROOT: {SHARE_ROOT}')

cmd = [sys.executable, '-m', 'pip', 'install', '-e', str(CC_REPO), 'uvicorn', 'httpx']
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)
print('CircuitCollector installed into:', sys.executable)


In [ ]:
def read_api_log_tail(lines=80):
    if not api_log.exists():
        return '<API log file does not exist yet>'
    text = api_log.read_text(errors='replace')
    return '\n'.join(text.splitlines()[-lines:]) or '<API log is empty>'

def api_health(url, timeout=1):
    try:
        with urllib.request.urlopen(url + '/health', timeout=timeout) as response:
            return True, response.read().decode(), None
    except Exception as exc:
        return False, None, exc

def circuitcollector_api_pids():
    try:
        result = subprocess.run(['pgrep', '-af', 'CircuitCollector.api.main:app'], text=True, capture_output=True)
    except FileNotFoundError:
        return []
    pids = []
    this_pid = os.getpid()
    for line in result.stdout.splitlines():
        parts = line.strip().split(maxsplit=1)
        if not parts or not parts[0].isdigit():
            continue
        pid = int(parts[0])
        cmdline = parts[1] if len(parts) > 1 else ''
        if pid != this_pid and 'uvicorn' in cmdline and 'CircuitCollector.api.main:app' in cmdline:
            pids.append((pid, cmdline))
    return pids

def stop_old_circuitcollector_apis():
    old = circuitcollector_api_pids()
    if not old:
        print('No old CircuitCollector API processes found.')
        return
    print('Stopping old CircuitCollector API processes:')
    for pid, cmdline in old:
        print(f'  SIGTERM pid={pid} {cmdline}')
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
    deadline = time.time() + 5
    while time.time() < deadline:
        if not circuitcollector_api_pids():
            print('Old CircuitCollector API processes stopped.')
            return
        time.sleep(0.25)
    for pid, cmdline in circuitcollector_api_pids():
        print(f'  SIGKILL pid={pid} {cmdline}')
        try:
            os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    time.sleep(0.5)

env = os.environ.copy()
env['PATH'] = '/foss/tools/ngspice/bin:/foss/tools/bin:' + env.get('PATH', '')
env['LD_LIBRARY_PATH'] = '/foss/tools/ngspice/lib:' + env.get('LD_LIBRARY_PATH', '')
env['PYTHONPATH'] = str(CC_REPO) + os.pathsep + env.get('PYTHONPATH', '')
api_log = SHARE_ROOT / 'circuitcollector_api.log'

API_URL = f'http://127.0.0.1:{PREFERRED_API_PORT}'
stop_old_circuitcollector_apis()

log_fh = api_log.open('w')
proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'CircuitCollector.api.main:app', '--host', '127.0.0.1', '--port', str(PREFERRED_API_PORT)],
    cwd=str(CC_REPO), env=env, stdout=log_fh, stderr=subprocess.STDOUT,
)
time.sleep(2)
if proc.poll() is not None:
    print(read_api_log_tail())
    raise RuntimeError(f'CircuitCollector API exited immediately with code {proc.returncode}. See {api_log}')
print('Started API pid:', proc.pid, 'url:', API_URL, 'log:', api_log)


In [ ]:
last_error = None
for attempt in range(60):
    try:
        with urllib.request.urlopen(API_URL + '/health', timeout=2) as response:
            print('API healthy at', API_URL, response.read().decode())
            break
    except Exception as exc:
        last_error = exc
        if 'proc' in globals() and proc is not None and proc.poll() is not None:
            print(read_api_log_tail())
            raise RuntimeError(f'CircuitCollector API process exited with code {proc.returncode}. See {api_log}') from exc
        if attempt in (0, 4, 14, 29, 44):
            print(f'Waiting for API health at {API_URL}/health; last error: {type(exc).__name__}: {exc}')
        time.sleep(1)
else:
    print(read_api_log_tail())
    raise RuntimeError(f'CircuitCollector API did not become healthy at {API_URL}/health. Last error: {last_error}. See {api_log}')


## Quick Analog-Amplifier Smoke Test

This verifies the bundled GF180 op-amp backend with the 5T OTA example.


In [ ]:
import json, urllib.request, urllib.error

analog_payload = {
    'base_config_path': 'config/gf180mcuD/opamp/5tota_single_gf180.toml',
    'params': {
        'M1_L': 0.56, 'M1_WL_ratio': 3.0, 'M1_M': 2,
        'M3_L': 0.56, 'M3_WL_ratio': 2.0, 'M3_M': 1,
        'M4_L': 0.56, 'M4_WL_ratio': 2.0, 'M4_M': 1,
        'M5_L': 0.56, 'M5_WL_ratio': 2.0, 'M5_M': 2,
        'ibias': 3e-5,
        'measure_mismatch': False,
        'measure_noise': False,
        'measure_slew_rate': False,
        'measure_output_swing': False,
    },
}

req = urllib.request.Request(
    API_URL + '/simulate/',
    data=json.dumps(analog_payload).encode(),
    headers={'Content-Type': 'application/json'},
    method='POST',
)

try:
    with urllib.request.urlopen(req, timeout=120) as r:
        analog_result = json.loads(r.read())
    print(json.dumps(analog_result, indent=2))
except urllib.error.HTTPError as e:
    body = e.read().decode()
    print('HTTP', e.code)
    print(body)
    raise


## Check RFPA Backend Coverage


In [ ]:
coverage = subprocess.run(
    ['python3', str(SKILL_ROOT / 'scripts' / 'check_backend_coverage.py'), '--backend-root', str(CC_ROOT)],
    check=True, capture_output=True, text=True,
)
print(coverage.stdout)


## Quick RFPA Smoke Test (Example Backend)


In [ ]:
smoke = subprocess.run(
    [
        sys.executable, str(SKILL_ROOT / 'scripts' / 'smoke_test_backend.py'),
        '--backend-root', str(CC_ROOT),
        '--api-url', API_URL + '/simulate/',
        '--quick',
        '--topology', 'two_stage_single_ended',
    ],
    check=False, capture_output=True, text=True,
)
print(smoke.stdout)
print(smoke.stderr)
